In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re


In [16]:
spark = SparkSession.builder \
    .appName("F1_Pipeline_Clean") \
    .getOrCreate()


In [17]:
pit = spark.read.option("header", True).csv("pitstop.csv")
safety = spark.read.option("header", True).csv("safety_cars.csv")
redflag = spark.read.option("header", True).csv("red_flags.csv")


In [22]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType
import re

# Normalization function
def normalize(text):
    return re.sub(r"[^a-z0-9]", "", text.lower())

normalize_udf = udf(normalize, StringType())

# Rename 'Race Name' -> 'Race' in all DataFrames
pit = pit.withColumnRenamed("Race Name", "Race")
safety = safety.withColumnRenamed("Race Name", "Race")
redflag = redflag.withColumnRenamed("Race Name", "Race")

# Add normalized race name column
pit = pit.withColumn("Race_norm", normalize_udf(col("Race")))
safety = safety.withColumn("Race_norm", normalize_udf(col("Race")))
redflag = redflag.withColumn("Race_norm", normalize_udf(col("Race")))


In [25]:
from pyspark.sql.functions import col, regexp_extract

pit = pit.withColumn("Pit_Time_num", col("Pit_Time").cast("double"))


# Convert stint numbers to integer
pit = pit.withColumn("Stint_num", regexp_extract(col("Stint"), r"(\d+)", 1).cast("int"))

# Extract only useful pitstop fields
pit_clean = pit.select( "Season", "Round", "Race_norm", "Driver", "Constructor", "Circuit", "Country","Stint_num", "Tire Compound", "Stint Length", "Pit_Time_num")

# Normalize driver name (anonymisation optional)
pit_clean = pit_clean.withColumn( "Driver_norm", regexp_extract(col("Driver"), r"(\w+)", 1))


In [26]:
stint_struct = struct(
    col("Stint_num").alias("stint"),
    col("Tire Compound").alias("tire"),
    col("Stint Length").alias("length"),
    col("Pit_Time_num").alias("pit_time")
)

group_cols = ["Season", "Round", "Race_norm", "Driver_norm", "Constructor"]

pit_grouped = pit_clean.groupBy(*group_cols).agg(
    collect_list(stint_struct).alias("stints"),
    avg("Pit_Time_num").alias("avg_pit_time"),
    sum("Pit_Time_num").alias("total_pit_time"),
    first("Circuit").alias("Circuit"),
    first("Country").alias("Country")
)


In [28]:
joined = pit_grouped \
    .join(broadcast(safety), "Race_norm", "left") \
    .join(broadcast(redflag), "Race_norm", "left")

# Boolean flags
joined = joined.withColumn("had_safety_car", col("FullLaps").isNotNull()) \
               .withColumn("had_red_flag", col("Lap").isNotNull())


In [29]:
def flatten(stints):
    if stints is None:
        return ""
    # Convert Row objects → Python dict → sorted by stint number
    s_sorted = sorted([s.asDict() for s in stints], key=lambda x: x["stint"])
    return ",".join([f"{s['stint']}:{s['tire']}({s['length']})" for s in s_sorted])

flatten_udf = udf(flatten, StringType())

final = joined.withColumn("stints_serial", flatten_udf(col("stints")))


In [36]:
final.select(F.explode("stints").alias("stint_row")) \
     .printSchema()


root
 |-- stint_row: struct (nullable = false)
 |    |-- stint: integer (nullable = true)
 |    |-- tire: string (nullable = true)
 |    |-- length: string (nullable = true)
 |    |-- pit_time: double (nullable = true)



In [38]:
from pyspark.sql.types import DoubleType
import pyspark.sql.functions as F

stints_exploded = final.withColumn("stint_row", F.explode("stints"))

stints_clean = stints_exploded.withColumn(
    "stint_numeric",
    F.when(F.col("stint_row.stint").rlike("^[0-9]+$"), F.col("stint_row.stint").cast(DoubleType()))
     .otherwise(None)
)


In [42]:
from pyspark.sql.types import DoubleType
import pyspark.sql.functions as F

# Explode the array if needed
stints_exploded = final.withColumn("stint_row", F.explode("stints"))

# Extract the stint column safely
stints_clean = stints_exploded.withColumn("stint_str", F.col("stint_row.stint"))

# Step: convert only numeric values; others become NULL
stints_clean = stints_clean.withColumn(
    "stint_numeric",
    F.when(F.col("stint_str").rlike(r"^\d+(\.\d+)?$"), F.col("stint_str").cast(DoubleType()))
     .otherwise(None)
)

# Drop unnecessary columns
stints_clean = stints_clean.drop("stint_row", "stints", "stint_str")

# Show result
stints_clean.show(truncate=False)


25/11/19 15:46:37 ERROR Executor: Exception in task 0.0 in stage 18.0 (TID 18)
org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'Final Stint' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 3 in cell [25]

	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:145)
	at org.apache.spark.sql.errors.QueryExecutionErrors.invalidInputInCastToNumberError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator

NumberFormatException: [CAST_INVALID_INPUT] The value 'Final Stint' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 3 in cell [25]


25/11/19 15:46:37 WARN TaskSetManager: Lost task 0.0 in stage 20.0 (TID 20) (10.33.74.20 executor driver): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 20 cancelled The corresponding SQL query has failed. SQLSTATE: XXKDA)
